**VM Scale Sets (VMSS)** let you deploy and manage a fleet of identical virtual machines that can automatically scale in and out based on demand or a schedule. They are the foundation of scalable, highly available compute in Azure — the equivalent of AWS Auto Scaling Groups combined with a launch template.

VMSS also integrates with Availability Zones to distribute instances across failure boundaries, with Azure Load Balancer or Application Gateway for traffic distribution, and with Azure Monitor for metric-driven autoscaling.

## Core Concepts

| Setting | Description |
|---|---|
| **Image** | The OS image all instances share |
| **VM size** | The size (CPU/memory) of every instance in the set |
| **Instance count** | Current number of running instances |
| **Min / Max** | Bounds for autoscaling |
| **Orchestration mode** | How Azure manages instance placement — Flexible or Uniform |
| **Upgrade policy** | How VM image updates are rolled out — Automatic, Rolling, or Manual |
| **Scale-in policy** | Which instances to remove first when scaling in |
| **Health extension / probe** | How VMSS detects unhealthy instances for automatic repair |

```
VM Scale Set
├── VM instance 0   (Zone 1)
├── VM instance 1   (Zone 2)
├── VM instance 2   (Zone 3)
├── VM instance 3   (Zone 1)
└── VM instance 4   (Zone 2)  ← new instance added by autoscale
```

## Orchestration Modes

VMSS has two orchestration modes that control how Azure places and manages instances:

### Flexible Orchestration (recommended)

- Azure places instances across **Availability Zones and fault domains** automatically
- Instances are **full Azure VMs** — each appears as a first-class resource in the portal and CLI
- You can mix VM sizes and use different configurations per instance
- You can manually add existing VMs to the scale set
- Supports **up to 1,000 instances** per scale set
- Works with all Azure load balancing services
- **Preferred for new designs** — combines the fleet management of VMSS with the flexibility of individual VMs

### Uniform Orchestration (legacy)

- All instances are **identical** — same image, same size, same configuration
- Instances are managed as a group — not individually addressable as standalone VMs
- Supports **up to 1,000 instances** (with platform image) or 600 (with custom image)
- Overprovisioning: Azure can launch extra instances during scale-out and delete the slowest ones — speeds up provisioning
- Required for some specific features (e.g. Spot instance mix with `SpotMixPolicy`)

> Microsoft recommends **Flexible orchestration** for most new workloads. Uniform is retained for backwards compatibility and specific Spot scenarios.

## Availability Sets (Deep Dive)

An **Availability Set** groups VMs within a single data centre across two dimensions to protect against hardware failures and planned maintenance.

### Fault Domains

A **Fault Domain (FD)** is a group of hardware that shares a common power source and network switch — a physical rack. If a rack loses power or the switch fails, all VMs in that fault domain are affected.

By spreading VMs across **up to 3 fault domains**, you ensure that a single rack failure takes down at most one-third of your fleet.

```
Availability Set
├── Fault Domain 0  →  VM-A, VM-D   (Rack 1 — separate power + switch)
├── Fault Domain 1  →  VM-B, VM-E   (Rack 2)
└── Fault Domain 2  →  VM-C, VM-F   (Rack 3)
```

### Update Domains

An **Update Domain (UD)** is a logical group of VMs that Azure reboots together during planned maintenance (host OS patching, hardware updates). Only one UD is rebooted at a time, and Azure waits between each.

Up to **20 update domains** per availability set. With 3 VMs across 3 UDs, at most 1 VM reboots during any maintenance window.

```
Availability Set
├── Update Domain 0  →  VM-A   (rebooted during maintenance window 1)
├── Update Domain 1  →  VM-B   (rebooted during maintenance window 2)
└── Update Domain 2  →  VM-C   (rebooted during maintenance window 3)
```

### Limitations
- All VMs must be in the **same region and same data centre**
- Cannot change a VM's availability set after creation — you must redeploy
- Protects within one building only — not against a whole-zone failure
- SLA: **99.95%** uptime for 2+ VMs in an Availability Set with Premium SSD

> For new designs, prefer **Availability Zones** — they protect against entire data centre failures and carry a 99.99% SLA.

## Availability Zones with VMSS

When you configure a VMSS with Availability Zones, Azure spreads instances across all selected zones automatically.

```
VMSS (3 zones selected)
├── Zone 1  →  Instance 0, Instance 3, Instance 6
├── Zone 2  →  Instance 1, Instance 4, Instance 7
└── Zone 3  →  Instance 2, Instance 5, Instance 8
```

### Zone balancing

VMSS tries to keep instances evenly distributed across zones. If one zone is unavailable during scale-out, Azure continues deploying to the remaining zones rather than failing the whole operation — this is called **best-effort zone balancing**.

With **strict zone balancing** enabled, VMSS will fail a scale-out if it cannot achieve even distribution — use this only when exact balance is critical.

### SLAs with zones

| Configuration | SLA |
|---|---|
| Single VM (Premium SSD) | 99.9% |
| Availability Set (2+ VMs) | 99.95% |
| Availability Zones (2+ VMs across 2+ zones) | 99.99% |

## Autoscaling

VMSS autoscaling adjusts the instance count based on metrics, schedules, or both.

### Metric-based scaling

Define **scale-out** and **scale-in** rules using Azure Monitor metrics:

```
Scale-out rule:  CPU > 70% for 5 minutes  →  add 2 instances
Scale-in rule:   CPU < 30% for 10 minutes →  remove 1 instance
```

Common metrics for scaling:

| Metric | Use case |
|---|---|
| **CPU percentage** | General workloads |
| **Memory percentage** | Memory-bound services |
| **Network in/out bytes** | High-throughput services |
| **Custom metric (App Insights)** | HTTP queue depth, request count, latency |
| **Service Bus queue length** | Background job workers |

### Scheduled scaling

Set minimum, maximum, and default instance counts at specific times.

```
Weekdays 08:00 UTC  →  min=5,  default=10
Weekdays 20:00 UTC  →  min=2,  default=3
```

Use for known traffic patterns — business hours, batch windows, end-of-month peaks.

### Cooldown period

After a scale-out event, the autoscaler waits for the **cooldown period** (default 5 minutes) before evaluating the next scale-out trigger. This prevents thrashing — repeated rapid scaling.

### Scale-in policy

When scaling in, VMSS decides which instances to remove based on the scale-in policy:

| Policy | Behaviour |
|---|---|
| **Default** | Balance across zones first; then remove the highest-numbered instance ID |
| **Newest VM** | Remove the most recently created instance |
| **Oldest VM** | Remove the oldest instance (useful for phasing out old image versions) |

## Upgrade Policies

When you update the VMSS model — changing the OS image, VM size, or configuration — the **upgrade policy** controls how that change is rolled out to existing instances.

| Policy | Behaviour | Use case |
|---|---|---|
| **Automatic** | Azure updates all instances immediately, in batches | Stateless, tolerant of disruption |
| **Rolling** | Instances updated in configurable batches with health checks between batches | Balanced — production workloads |
| **Manual** | Instances are not updated until you explicitly trigger it | Full control — databases, stateful apps |

### Rolling upgrade in detail

Rolling upgrades update a configurable **batch percentage** of instances at a time, pausing between batches to verify health:

```
Batch 1 (20% of fleet) → health check passes → continue
Batch 2 (20% of fleet) → health check passes → continue
...                    → health check FAILS  → pause and alert
```

Key settings:
- **Max batch instance percent** — how large each batch is (e.g. 20%)
- **Max unhealthy instance percent** — how many instances can be unhealthy before aborting
- **Max unhealthy upgraded instance percent** — threshold for considering the upgrade failed
- **Pause between batches** — wait time between each batch (seconds)

> Rolling upgrade requires a **health extension or Application Health probe** to be configured — otherwise Azure cannot determine if instances are healthy after upgrade.

## Instance Health & Automatic Repair

VMSS can automatically detect and replace unhealthy instances using either:

- **Application Health Extension** — runs inside the VM and reports health via a local HTTP endpoint
- **Load Balancer health probe** — the probe configured on the attached Azure Load Balancer

### Automatic Repair

When enabled, if an instance is unhealthy for longer than the **grace period** (default 30 minutes), VMSS:
1. Deletes the unhealthy instance
2. Launches a new replacement instance

```
Instance becomes unhealthy
  └── Wait grace period (e.g. 30 min)
        └── Still unhealthy?
              └── Delete → Launch replacement
```

The grace period prevents premature replacement of instances that are still bootstrapping or temporarily overloaded.

### Repair actions (preview)

Instead of immediately deleting an unhealthy instance, you can configure a **repair action** — for example, reimaging (reinstalling the OS) rather than deleting and recreating.

## Spot Instances in VMSS

VMSS supports **Azure Spot VMs** within the fleet, either as 100% Spot or as a **mixed fleet** (Spot + on-demand).

### Mixed fleet with Spot priority

Configure a VMSS to try Spot first, then fall back to regular (on-demand) instances if Spot capacity is unavailable:

```
Try to provision Spot → capacity unavailable → provision Regular
```

This is useful for batch workloads where you want maximum cost savings but cannot tolerate scale-out failures.

### Eviction policy for Spot in VMSS

| Policy | Effect on eviction |
|---|---|
| **Deallocate** | Instance is stopped and kept in the scale set — can restart when Spot capacity returns |
| **Delete** | Instance is deleted from the scale set — reduces instance count |

> Use **Deallocate** if you want to resume Spot instances automatically when capacity returns. Use **Delete** if you want the scale set to replace them with new instances.

## Proximity Placement Groups

A **Proximity Placement Group (PPG)** is an Azure construct that co-locates VMs and VMSS instances within the same data centre — minimising network latency between them.

```
Without PPG: VMs may land in different buildings within the same zone
With PPG:    All VMs land in the same physical building
```

| | AWS equivalent | Use case |
|---|---|---|
| Proximity Placement Group | Cluster Placement Group | Tightly coupled workloads — HPC, low-latency app tiers, SAP systems |

### Trade-offs
- Lower latency between VMs in the PPG
- Reduces available hardware capacity — PPG is constrained to one physical cluster
- Cannot use Availability Zones and guarantee all VMs land in the PPG simultaneously — one of the first VMs "anchors" the PPG to a specific host cluster

> Use PPGs for latency-sensitive multi-tier apps (web + app + cache all in one PPG). Do not use for general workloads — the capacity constraints are not worth it.

## Working with VMSS via Python

In [ ]:
# pip install azure-identity azure-mgmt-compute azure-mgmt-monitor

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.compute import ComputeManagementClient
from azure.mgmt.monitor import MonitorManagementClient

credential = DefaultAzureCredential()
subscription_id = "<your-subscription-id>"
resource_group  = "my-rg"

compute = ComputeManagementClient(credential, subscription_id)

In [ ]:
# Create a VMSS with Flexible orchestration across 3 availability zones
vmss_params = {
    "location": "eastus",
    "sku": {"name": "Standard_D2s_v5", "tier": "Standard", "capacity": 3},
    "orchestration_mode": "Flexible",
    "platform_fault_domain_count": 1,  # 1 = spread across zones
    "zones": ["1", "2", "3"],
    "virtual_machine_profile": {
        "os_profile": {
            "computer_name_prefix": "myvm",
            "admin_username": "azureuser",
        },
        "storage_profile": {
            "image_reference": {
                "publisher": "Canonical",
                "offer": "0001-com-ubuntu-server-jammy",
                "sku": "22_04-lts-gen2",
                "version": "latest"
            },
            "os_disk": {"create_option": "FromImage", "managed_disk": {"storage_account_type": "Premium_LRS"}}
        },
        "network_profile": {
            "network_api_version": "2020-11-01",
            "network_interface_configurations": [{
                "name": "mynic",
                "primary": True,
                "ip_configurations": [{"name": "myipconfig", "subnet": {"id": "<subnet-resource-id>"}}]
            }]
        }
    }
}

poller = compute.virtual_machine_scale_sets.begin_create_or_update(resource_group, "my-vmss", vmss_params)
vmss = poller.result()
print(f"VMSS created: {vmss.name}")

In [ ]:
# List all instances in a VMSS and their power state
instances = compute.virtual_machine_scale_set_vms.list(
    resource_group, "my-vmss", expand="instanceView"
)
for inst in instances:
    statuses = inst.instance_view.statuses if inst.instance_view else []
    power = next((s.display_status for s in statuses if "VM" in (s.display_status or "")), "unknown")
    print(f"Instance {inst.instance_id:<4} {inst.name:<30} {power}")

In [ ]:
# Manually scale the VMSS to 5 instances
vmss_obj = compute.virtual_machine_scale_sets.get(resource_group, "my-vmss")
vmss_obj.sku.capacity = 5

poller = compute.virtual_machine_scale_sets.begin_create_or_update(resource_group, "my-vmss", vmss_obj)
poller.result()
print("Scaled to 5 instances")

In [ ]:
from azure.mgmt.monitor import MonitorManagementClient
from azure.mgmt.monitor.models import AutoscaleSettingResource, AutoscaleProfile, ScaleCapacity, ScaleRule, MetricTrigger, ScaleAction
from datetime import timedelta

monitor = MonitorManagementClient(credential, subscription_id)

vmss_id = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Compute/virtualMachineScaleSets/my-vmss"

autoscale = AutoscaleSettingResource(
    location="eastus",
    enabled=True,
    target_resource_uri=vmss_id,
    profiles=[
        AutoscaleProfile(
            name="cpu-autoscale",
            capacity=ScaleCapacity(minimum="2", maximum="10", default="3"),
            rules=[
                # Scale-out: CPU > 70% for 5 min → add 2 instances
                ScaleRule(
                    metric_trigger=MetricTrigger(
                        metric_name="Percentage CPU",
                        metric_resource_uri=vmss_id,
                        time_grain=timedelta(minutes=1),
                        statistic="Average",
                        time_window=timedelta(minutes=5),
                        time_aggregation="Average",
                        operator="GreaterThan",
                        threshold=70
                    ),
                    scale_action=ScaleAction(direction="Increase", type="ChangeCount", value="2", cooldown=timedelta(minutes=5))
                ),
                # Scale-in: CPU < 30% for 10 min → remove 1 instance
                ScaleRule(
                    metric_trigger=MetricTrigger(
                        metric_name="Percentage CPU",
                        metric_resource_uri=vmss_id,
                        time_grain=timedelta(minutes=1),
                        statistic="Average",
                        time_window=timedelta(minutes=10),
                        time_aggregation="Average",
                        operator="LessThan",
                        threshold=30
                    ),
                    scale_action=ScaleAction(direction="Decrease", type="ChangeCount", value="1", cooldown=timedelta(minutes=5))
                )
            ]
        )
    ]
)

monitor.autoscale_settings.create_or_update(resource_group, "my-vmss-autoscale", autoscale)
print("Autoscale settings applied")

## Summary

| Concept | Key Takeaway |
|---|---|
| VMSS | Fleet of identical VMs that scales automatically — Azure's Auto Scaling Group |
| Flexible orchestration | Recommended — instances are full VMs; spreads across zones and fault domains automatically |
| Uniform orchestration | Legacy — identical instances managed as a group; required for some Spot mix scenarios |
| Availability Set | FDs (rack isolation) + UDs (maintenance stagger) within one data centre — 99.95% SLA |
| Availability Zones | Spread across physically separate buildings — 99.99% SLA; preferred for new designs |
| Metric-based autoscaling | Scale in/out based on CPU, memory, custom metrics — define scale-out and scale-in rules |
| Scheduled autoscaling | Set capacity at specific times for known traffic patterns |
| Scale-in policy | Default (zone balance first), Newest VM, or Oldest VM |
| Upgrade policy | Automatic (immediate), Rolling (batched with health checks), Manual (explicit trigger) |
| Rolling upgrade | Configurable batch size and health thresholds — requires health extension or probe |
| Automatic repair | Detects and replaces unhealthy instances after a configurable grace period |
| Spot in VMSS | Mixed fleet — Spot first, fall back to regular; eviction: Deallocate or Delete |
| Proximity Placement Group | Co-locate VMs in the same physical cluster for lowest latency — like AWS Cluster PG |